# CEG-WM Colab 论文级单变量消融 Notebook

本 Notebook 用于同对象、同上游产物、只切一个 detect-side 开关的论文级单变量消融。

工作流约束：
1. 只执行一次 embed。
2. 所有 variant 复用同一份 base_embed/records/embed_record.json。
3. 仅重跑 detect-side variant，不改正式方法逻辑。
4. compare 结果统一写入 compare/ablation_manifest.json、compare/ablation_compare_summary.json、compare/ablation_compare_table.csv。
5. 运行结果目录和 ZIP 可同步保存到 Google Drive。

本 Notebook 只负责 Colab 流程编排；复杂逻辑继续下沉到 scripts/run_paper_ablation_workflow.py。

## 阶段概览

本 Notebook 尽量继承 Paper_Full_Cuda.ipynb 的阶段组织，但主流程执行入口切换为 paper ablation workflow。

阶段顺序：
1. 环境初始化
2. Google Drive 挂载
3. 仓库来源模式与定位
4. 依赖安装
5. 模型与权重准备
6. attestation 环境变量准备
7. ablation precheck
8. 主流程执行
9. compare summary 展示
10. 结果打包与同步 Drive

In [ ]:
import os
import sys
from pathlib import Path

import psutil

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("=" * 80)
print("初始化 Notebook 环境")
print("=" * 80)

if IN_COLAB:
    WORK_DIR = Path("/content")
    print("✓ 检测到 Google Colab 环境")
else:
    WORK_DIR = Path.cwd()
    print("✓ 当前为本地 Jupyter 环境")

WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"工作目录：{WORK_DIR}")
print(f"Python 版本：{sys.version.split()[0]}")
print(f"CPU 核心数：{psutil.cpu_count()}")
print(f"可用内存：{psutil.virtual_memory().available / (1024 ** 3):.1f} GB")

In [ ]:
from pathlib import Path

print("=" * 80)
print("挂载 Google Drive")
print("=" * 80)

DRIVE_MOUNT_POINT = Path("/content/drive")
GOOGLE_DRIVE_ROOT = None
GOOGLE_DRIVE_EXPORT_DIR = None

if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
        GOOGLE_DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
        GOOGLE_DRIVE_EXPORT_DIR = GOOGLE_DRIVE_ROOT / "CEG_WM_Outputs" / "Paper_Ablation_Cuda"
        GOOGLE_DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"✓ Google Drive 已挂载：{GOOGLE_DRIVE_ROOT}")
        print(f"✓ 默认导出目录：{GOOGLE_DRIVE_EXPORT_DIR}")
    except Exception as exc:
        GOOGLE_DRIVE_ROOT = None
        GOOGLE_DRIVE_EXPORT_DIR = None
        print(f"✗ Google Drive 挂载失败：{exc}")
else:
    print("ℹ 当前不是 Google Colab 环境，跳过 Google Drive 挂载")

In [ ]:
import shutil
import subprocess
import zipfile
from pathlib import Path

REPO_SOURCE_MODE = "git_clone"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "main"
TARGET_DIR = WORK_DIR / "CEG-WM"

print("=" * 80)
print("仓库来源模式配置")
print("=" * 80)
print("当前固定模式：git_clone")
print(f"REPO_URL：{REPO_URL}")
print(f"REPO_BRANCH：{REPO_BRANCH}")
print(f"TARGET_DIR：{TARGET_DIR}")

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path


def _print_command_tail(label: str, text: str, max_lines: int = 20) -> None:
    print(f"\n{label}（最后 {max_lines} 行）：")
    lines = (text or "").splitlines()
    print("\n".join(lines[-max_lines:]) if lines else "<empty>")


def _run_checked_command(command, cwd=None):
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if result.returncode != 0:
        _print_command_tail("STDOUT", result.stdout)
        _print_command_tail("STDERR", result.stderr)
        raise RuntimeError(
            f"命令执行失败，return_code={result.returncode}，command={' '.join(command)}"
        )
    return result


def find_ceg_wm_root(base_dir: Path) -> Path:
    characteristic_paths = ["main/cli", "configs", "scripts", "requirements.txt"]
    if all((base_dir / item).exists() for item in characteristic_paths):
        return base_dir
    for subdir in sorted(base_dir.iterdir()):
        if subdir.is_dir() and all((subdir / item).exists() for item in characteristic_paths):
            return subdir
    raise FileNotFoundError(
        f"无法在 {base_dir} 下定位 CEG-WM 根目录，请检查 git_clone 获取结果。"
    )


print("=" * 80)
print("清理当前目录并重新拉取仓库")
print("=" * 80)

if shutil.which("git") is None:
    raise RuntimeError("当前环境未检测到 git 命令，请先安装 git")

if TARGET_DIR.exists():
    print(f"检测到现有目录，先删除：{TARGET_DIR}")
    shutil.rmtree(TARGET_DIR)

_run_checked_command(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(TARGET_DIR)])

CEG_WM_ROOT = find_ceg_wm_root(TARGET_DIR)
print(f"CEG_WM_ROOT：{CEG_WM_ROOT}")
print("✓ 已完成清理并重新 git clone 最新仓库代码")

In [ ]:
import subprocess
import sys

print("=" * 80)
print("安装依赖")
print("=" * 80)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], capture_output=True)

if IN_COLAB:
    subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"], capture_output=True)
    print("✓ Colab 系统依赖已安装")

pyproject_file = CEG_WM_ROOT / "pyproject.toml"
requirements_file = CEG_WM_ROOT / "requirements.txt"
if pyproject_file.exists():
    install_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", str(CEG_WM_ROOT)],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
elif requirements_file.exists():
    install_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
else:
    raise FileNotFoundError("未找到 pyproject.toml 或 requirements.txt")

if install_result.returncode != 0:
    print(install_result.stderr[-500:])
    raise RuntimeError("CEG-WM 依赖安装失败")

transparent_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "transparent-background"],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
if transparent_result.returncode == 0:
    print("✓ transparent-background 安装成功")
else:
    print("⚠ transparent-background 安装失败，内容语义掩码可能退化为 fallback")

print("✓ 依赖安装完成")

In [ ]:
import gc
import os

import torch
from huggingface_hub import HfApi, scan_cache_dir, snapshot_download

print("=" * 80)
print("下载 Stable Diffusion 模型")
print("=" * 80)

MODEL_CACHE_DIR = WORK_DIR / "huggingface_cache"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(MODEL_CACHE_DIR)

MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"

try:
    api = HfApi()
    try:
        user_info = api.whoami()
        print(f"✓ Hugging Face 已认证：{user_info.get('name', 'Unknown')}")
    except Exception:
        print("ℹ 当前使用匿名 Hugging Face 访问")
except Exception:
    print("ℹ 无法确认 Hugging Face 认证状态")


def _model_cached(model_id: str) -> bool:
    try:
        cache_info = scan_cache_dir()
        return any(model_id in repo.repo_id for repo in cache_info.repos)
    except Exception:
        return False

if _model_cached(MODEL_ID):
    print(f"✓ 模型已缓存：{MODEL_ID}")
else:
    snapshot_path = snapshot_download(
        repo_id=MODEL_ID,
        cache_dir=str(MODEL_CACHE_DIR),
        local_files_only=False,
    )
    print(f"✓ 模型下载完成：{snapshot_path}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✓ 模型准备完成")

In [ ]:
import os
import shutil
import urllib.request
from pathlib import Path

print("=" * 80)
print("下载 InSPyReNet 权重")
print("=" * 80)

repo_inspyrenet_model_path = CEG_WM_ROOT / "models" / "inspyrenet" / "ckpt_base.pth"
cached_inspyrenet_model_path = WORK_DIR / "models" / "inspyrenet" / "ckpt_base.pth"
repo_inspyrenet_model_path.parent.mkdir(parents=True, exist_ok=True)
cached_inspyrenet_model_path.parent.mkdir(parents=True, exist_ok=True)

default_inspyrenet_url = "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth"
inspyrenet_model_url = os.environ.get("INSPYRENET_MODEL_URL", default_inspyrenet_url).strip()


def _is_valid_weight_file(path: Path) -> bool:
    return path.exists() and path.is_file() and path.stat().st_size > 0


existing_path = None
for candidate in [repo_inspyrenet_model_path, cached_inspyrenet_model_path]:
    if _is_valid_weight_file(candidate):
        existing_path = candidate
        break

if existing_path is None:
    temp_download_path = cached_inspyrenet_model_path.with_suffix(".downloading")
    if temp_download_path.exists():
        temp_download_path.unlink()
    urllib.request.urlretrieve(inspyrenet_model_url, str(temp_download_path))
    if not _is_valid_weight_file(temp_download_path):
        temp_download_path.unlink(missing_ok=True)
        raise RuntimeError(f"下载完成但权重文件无效：{temp_download_path}")
    temp_download_path.replace(cached_inspyrenet_model_path)
    existing_path = cached_inspyrenet_model_path

if existing_path.resolve() != repo_inspyrenet_model_path.resolve():
    shutil.copy2(existing_path, repo_inspyrenet_model_path)

print(f"✓ InSPyReNet 权重路径：{repo_inspyrenet_model_path}")
print(f"✓ 权重大小：{repo_inspyrenet_model_path.stat().st_size} bytes")

In [ ]:
from datetime import datetime
from pathlib import Path

RUN_TAG = datetime.now().strftime("paper_ablation_%Y%m%d_%H%M%S")
SELECTED_VARIANTS = ["GEO-off", "GEO-on"]
FRESH_RUN = True
RESUME = False
REUSE_BASE_EMBED_RECORD = ""
PACKAGE_ZIP = True
SYNC_TO_DRIVE = True
DRIVE_EXPORT_SUBDIR = "Paper_Ablation_Cuda"

print("=" * 80)
print("实验定义")
print("=" * 80)
print(f"RUN_TAG = {RUN_TAG}")
print(f"SELECTED_VARIANTS = {SELECTED_VARIANTS}")
print(f"FRESH_RUN = {FRESH_RUN}")
print(f"RESUME = {RESUME}")
print(f"REUSE_BASE_EMBED_RECORD = {REUSE_BASE_EMBED_RECORD or '<empty>'}")
print(f"PACKAGE_ZIP = {PACKAGE_ZIP}")
print(f"SYNC_TO_DRIVE = {SYNC_TO_DRIVE}")
print(f"DRIVE_EXPORT_SUBDIR = {DRIVE_EXPORT_SUBDIR}")

## 单变量说明

本轮默认比较 GEO repair detect-side 开关，而不是关闭整个 geometry 链。

要求：
1. GEO-on 仅设置 detect.geometry.geo_score_repair.enabled = true。
2. GEO-off 仅设置 detect.geometry.geo_score_repair.enabled = false。
3. geometry chain 本身保持开启，不修改 fusion、decision、threshold 和正式得分定义。

In [ ]:
ABLATION_SWITCH_NAME = "detect.geometry.geo_score_repair.enabled"
SWITCH_ON_VARIANT = "GEO-on"
SWITCH_OFF_VARIANT = "GEO-off"
SINGLE_VARIABLE_REASON = (
    "只切 detect.geometry.geo_score_repair.enabled 这一个 detect-side 开关；"
    "geometry chain 保持 enabled，不关闭 attention anchor、sync、fusion、decision、threshold。"
)

print("=" * 80)
print("单变量说明")
print("=" * 80)
print(f"ABLATION_SWITCH_NAME = {ABLATION_SWITCH_NAME}")
print(f"SWITCH_ON_VARIANT = {SWITCH_ON_VARIANT}")
print(f"SWITCH_OFF_VARIANT = {SWITCH_OFF_VARIANT}")
print(f"SINGLE_VARIABLE_REASON = {SINGLE_VARIABLE_REASON}")

In [ ]:
import json
import os
import re
import secrets
from datetime import datetime

print("=" * 80)
print("准备 attestation 环境变量")
print("=" * 80)

ATTESTATION_ENV_SPECS = {
    "CEG_WM_K_MASTER": 64,
    "CEG_WM_K_PROMPT": 32,
    "CEG_WM_K_SEED": 32,
}
USER_ATTESTATION_VALUES = {
    "CEG_WM_K_MASTER": "",
    "CEG_WM_K_PROMPT": "",
    "CEG_WM_K_SEED": "",
}
AUTO_GENERATE_MISSING_ATTESTATION_KEYS = True
ATTESTATION_INFO_PATH = WORK_DIR / "attestation_env_keys.json"


def _is_valid_hex_secret(value: str, expected_length: int) -> bool:
    candidate = value.strip().lower()
    return len(candidate) == expected_length and re.fullmatch(r"[0-9a-f]+", candidate) is not None


def _mask_secret(value: str) -> str:
    if len(value) <= 8:
        return "*" * len(value)
    return f"{value[:4]}...{value[-4:]}"


resolved_values = {}
generated_env_vars = []
missing_env_vars = []
for env_name, expected_length in ATTESTATION_ENV_SPECS.items():
    manual_value = USER_ATTESTATION_VALUES.get(env_name, "")
    existing_value = os.environ.get(env_name, "")
    if _is_valid_hex_secret(manual_value, expected_length):
        resolved_value = manual_value.strip().lower()
        source = "manual_input"
    elif _is_valid_hex_secret(existing_value, expected_length):
        resolved_value = existing_value.strip().lower()
        source = "existing_env"
    elif AUTO_GENERATE_MISSING_ATTESTATION_KEYS:
        resolved_value = secrets.token_hex(expected_length // 2)
        source = "generated_for_session"
        generated_env_vars.append(env_name)
    else:
        resolved_value = ""
        source = "missing"

    if resolved_value:
        os.environ[env_name] = resolved_value
        resolved_values[env_name] = {
            "value": resolved_value,
            "masked_value": _mask_secret(resolved_value),
            "length": len(resolved_value),
            "source": source,
        }
    else:
        missing_env_vars.append(env_name)

if missing_env_vars:
    raise RuntimeError(f"以下 attestation 环境变量未就绪：{missing_env_vars}")

ATTESTATION_INFO_PATH.write_text(
    json.dumps(
        {
            "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "resolved_values": resolved_values,
            "generated_env_vars": generated_env_vars,
            "env_specs": ATTESTATION_ENV_SPECS,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

for env_name, payload in resolved_values.items():
    print(f"✓ {env_name}: source={payload['source']}, value={payload['masked_value']}")
print(f"✓ attestation 信息已写入：{ATTESTATION_INFO_PATH}")

In [ ]:
import socket

import torch
import yaml

print("=" * 80)
print("ablation precheck")
print("=" * 80)

ABLATION_CONFIG_PATH = CEG_WM_ROOT / "configs" / "ablation" / "paper_ablation_cuda.yaml"
ABLATION_SCRIPT_PATH = CEG_WM_ROOT / "scripts" / "run_paper_ablation_workflow.py"
precheck_results = []


def _record_check(name: str, ok: bool, detail: str) -> None:
    precheck_results.append({"name": name, "ok": ok, "detail": detail})
    print(f"{'✓' if ok else '✗'} {name}: {detail}")


_record_check("ablation config 存在", ABLATION_CONFIG_PATH.exists(), str(ABLATION_CONFIG_PATH))
_record_check("workflow 脚本存在", ABLATION_SCRIPT_PATH.exists(), str(ABLATION_SCRIPT_PATH))
_record_check("CUDA 可用", bool(torch.cuda.is_available()), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "未检测到 GPU")
_record_check(
    "FRESH_RUN 与 RESUME 不冲突",
    not (bool(FRESH_RUN) and bool(RESUME)),
    f"FRESH_RUN={FRESH_RUN}, RESUME={RESUME}",
)

reuse_record_path = Path(REUSE_BASE_EMBED_RECORD).expanduser() if str(REUSE_BASE_EMBED_RECORD).strip() else None
_record_check(
    "复用 embed_record 路径有效",
    reuse_record_path is None or reuse_record_path.exists(),
    str(reuse_record_path) if reuse_record_path is not None else "<not used>",
)

with open(ABLATION_CONFIG_PATH, "r", encoding="utf-8") as handle:
    ablation_cfg = yaml.safe_load(handle)
workflow_cfg = ablation_cfg.get("paper_ablation_workflow", {}) if isinstance(ablation_cfg, dict) else {}
detect_rerun_cfg = workflow_cfg.get("detect_rerun", {}) if isinstance(workflow_cfg, dict) else {}
variant_defs = detect_rerun_cfg.get("variants", []) if isinstance(detect_rerun_cfg, dict) else []
available_variants = [item.get("name") for item in variant_defs if isinstance(item, dict)]
missing_variants = [name for name in SELECTED_VARIANTS if name not in available_variants]
_record_check("selected variants 合法", len(missing_variants) == 0, f"missing={missing_variants}")
_record_check(
    "默认 GEO on/off 为 detect-side 单变量",
    all(name in available_variants for name in [SWITCH_ON_VARIANT, SWITCH_OFF_VARIANT]),
    f"available={available_variants}",
)

for env_name, expected_length in ATTESTATION_ENV_SPECS.items():
    value = os.environ.get(env_name, "").strip().lower()
    ok = len(value) == expected_length and all(ch in "0123456789abcdef" for ch in value)
    _record_check(f"attestation 环境变量: {env_name}", ok, f"len={len(value)}")

if SYNC_TO_DRIVE:
    _record_check(
        "Google Drive 导出目录可用",
        GOOGLE_DRIVE_EXPORT_DIR is not None,
        str(GOOGLE_DRIVE_EXPORT_DIR) if GOOGLE_DRIVE_EXPORT_DIR is not None else "<absent>",
    )

try:
    socket.setdefaulttimeout(15)
    from huggingface_hub import HfApi

    api = HfApi()
    _ = api.model_info("stabilityai/stable-diffusion-3.5-medium")
    _record_check("HF 模型可访问", True, "stabilityai/stable-diffusion-3.5-medium")
except Exception as exc:
    _record_check("HF 模型可访问", False, f"{type(exc).__name__}: {exc}")

hard_fail = [item for item in precheck_results if not item["ok"]]
if hard_fail:
    raise RuntimeError(f"ablation precheck 未通过，失败项数量={len(hard_fail)}")

print("✓ ablation precheck 全部通过")

In [ ]:
import json
import subprocess
import sys

import yaml

print("=" * 80)
print("执行 paper ablation workflow")
print("=" * 80)

runtime_config_dir = CEG_WM_ROOT / "outputs" / "notebook_runtime_configs"
runtime_config_dir.mkdir(parents=True, exist_ok=True)
runtime_config_path = runtime_config_dir / f"{RUN_TAG}.yaml"

with open(ABLATION_CONFIG_PATH, "r", encoding="utf-8") as handle:
    runtime_cfg = yaml.safe_load(handle)
if not isinstance(runtime_cfg, dict):
    raise TypeError("ablation 配置根节点必须是 dict")

mask_cfg = runtime_cfg.setdefault("mask", {})
if not isinstance(mask_cfg, dict):
    raise TypeError("mask 配置必须是 dict")
mask_cfg["semantic_model_path"] = str(repo_inspyrenet_model_path.resolve())

prompt_file_value = runtime_cfg.get("inference_prompt_file")
if isinstance(prompt_file_value, str) and prompt_file_value.strip():
    prompt_file_path = Path(prompt_file_value).expanduser()
    if not prompt_file_path.is_absolute():
        prompt_file_path = (CEG_WM_ROOT / prompt_file_path).resolve()
    else:
        prompt_file_path = prompt_file_path.resolve()
    runtime_cfg["inference_prompt_file"] = str(prompt_file_path)

workflow_cfg = runtime_cfg.setdefault("paper_ablation_workflow", {})
notebook_runtime_cfg = workflow_cfg.setdefault("notebook_runtime", {})
notebook_runtime_cfg["repo_source_mode"] = REPO_SOURCE_MODE
notebook_runtime_cfg["repo_url"] = REPO_URL
notebook_runtime_cfg["repo_branch"] = REPO_BRANCH
notebook_runtime_cfg["selected_variants"] = list(SELECTED_VARIANTS)
notebook_runtime_cfg["base_embed_reuse_mode"] = (
    "reuse_existing_record"
    if str(REUSE_BASE_EMBED_RECORD).strip()
    else "resume"
    if RESUME
    else "fresh_run"
    if FRESH_RUN
    else "new_run"
)
notebook_runtime_cfg["reuse_base_embed_record"] = str(REUSE_BASE_EMBED_RECORD).strip() or None
notebook_runtime_cfg["package_zip"] = bool(PACKAGE_ZIP)
notebook_runtime_cfg["sync_to_drive"] = bool(SYNC_TO_DRIVE)
notebook_runtime_cfg["export_to_drive"] = bool(SYNC_TO_DRIVE)
notebook_runtime_cfg["drive_export_subdir"] = DRIVE_EXPORT_SUBDIR
workflow_cfg["run_tag"] = RUN_TAG

runtime_config_path.write_text(
    yaml.safe_dump(runtime_cfg, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)
print(f"✓ 运行时配置已写入：{runtime_config_path}")
print(f"✓ semantic_model_path = {mask_cfg['semantic_model_path']}")
print(f"✓ inference_prompt_file = {runtime_cfg.get('inference_prompt_file')}")

command = [
    sys.executable,
    str(ABLATION_SCRIPT_PATH),
    "--config",
    str(runtime_config_path),
    "--run-tag",
    RUN_TAG,
]
for variant_name in SELECTED_VARIANTS:
    command.extend(["--variant", variant_name])
if FRESH_RUN:
    command.append("--fresh-run")
if RESUME:
    command.append("--resume")
if str(REUSE_BASE_EMBED_RECORD).strip():
    command.extend(["--reuse-base-embed-record", str(Path(REUSE_BASE_EMBED_RECORD).expanduser())])

print("执行命令：")
print(" ".join(command))

execution_result = subprocess.run(
    command,
    cwd=str(CEG_WM_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

if execution_result.stdout:
    print("\nSTDOUT（最后 40 行）：")
    print("\n".join(execution_result.stdout.splitlines()[-40:]))
if execution_result.stderr:
    print("\nSTDERR（最后 20 行）：")
    print("\n".join(execution_result.stderr.splitlines()[-20:]))
if execution_result.returncode != 0:
    raise RuntimeError(f"paper ablation workflow 执行失败，return_code={execution_result.returncode}")

WORKFLOW_RESULT = json.loads(execution_result.stdout)
RUN_ROOT = Path(WORKFLOW_RESULT["run_root"])
OUTPUT_ROOT = RUN_ROOT
COMPARE_SUMMARY_PATH = Path(WORKFLOW_RESULT["compare_summary_path"])
COMPARE_TABLE_PATH = Path(WORKFLOW_RESULT["compare_table_path"])
MANIFEST_PATH = Path(WORKFLOW_RESULT["manifest_path"])
ARCHIVE_PATH = Path(WORKFLOW_RESULT["archive_path"]) if WORKFLOW_RESULT.get("archive_path") else None

print("\n✓ workflow 执行完成")
print(f"RUN_ROOT = {RUN_ROOT}")
print(f"COMPARE_SUMMARY_PATH = {COMPARE_SUMMARY_PATH}")
print(f"COMPARE_TABLE_PATH = {COMPARE_TABLE_PATH}")
print(f"MANIFEST_PATH = {MANIFEST_PATH}")
print(f"ARCHIVE_PATH = {ARCHIVE_PATH if ARCHIVE_PATH is not None else '<none>'}")

In [ ]:
import json

try:
    import pandas as pd
except Exception:
    pd = None

print("=" * 80)
print("展示 compare summary")
print("=" * 80)

compare_summary = json.loads(COMPARE_SUMMARY_PATH.read_text(encoding="utf-8"))
compare_variants = compare_summary.get("variants", []) if isinstance(compare_summary, dict) else []
print(f"variant_count = {compare_summary.get('variant_count')}")
print(f"base_embed_record_path = {compare_summary.get('base_embed_record_path')}")

if pd is not None and compare_variants:
    compare_df = pd.DataFrame(compare_variants)
    preferred_columns = [
        "variant_name",
        "attestation_status",
        "content_attestation_score",
        "event_attestation_score",
        "channel_scores_lf",
        "channel_scores_hf",
        "channel_scores_geo",
        "active_geo_score_source",
        "geo_repair_enabled",
        "geo_repair_active",
        "geo_repair_mode",
        "geo_repair_direction_classification",
        "protocol_root_cause_classification",
        "base_embed_record_path",
        "detect_record_path",
    ]
    display_columns = [column for column in preferred_columns if column in compare_df.columns]
    display(compare_df[display_columns])
else:
    print(json.dumps(compare_summary, indent=2, ensure_ascii=False))

print(f"compare summary JSON = {COMPARE_SUMMARY_PATH}")
print(f"compare table CSV = {COMPARE_TABLE_PATH}")

In [ ]:
import shutil
from pathlib import Path

print("=" * 80)
print("打包结果并同步到 Google Drive")
print("=" * 80)

if PACKAGE_ZIP and ARCHIVE_PATH is None:
    archive_base = RUN_ROOT.parent / f"{RUN_ROOT.name}_paper_ablation_outputs"
    generated_archive = shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=str(RUN_ROOT.parent),
        base_dir=RUN_ROOT.name,
    )
    ARCHIVE_PATH = Path(generated_archive)

drive_run_dir = None
drive_zip_path = None
if SYNC_TO_DRIVE:
    if GOOGLE_DRIVE_EXPORT_DIR is None:
        raise RuntimeError("SYNC_TO_DRIVE=True，但当前 Google Drive 导出目录不可用")
    drive_root = GOOGLE_DRIVE_EXPORT_DIR / DRIVE_EXPORT_SUBDIR
    drive_root.mkdir(parents=True, exist_ok=True)
    drive_run_dir = drive_root / RUN_TAG
    if drive_run_dir.exists():
        shutil.rmtree(drive_run_dir)
    shutil.copytree(RUN_ROOT, drive_run_dir)
    if ARCHIVE_PATH is not None and ARCHIVE_PATH.exists():
        drive_zip_path = drive_root / ARCHIVE_PATH.name
        shutil.copy2(ARCHIVE_PATH, drive_zip_path)

print(f"本地结果目录 = {RUN_ROOT}")
print(f"本地 ZIP 路径 = {ARCHIVE_PATH if ARCHIVE_PATH is not None else '<none>'}")
print(f"Google Drive 目标目录 = {drive_run_dir if drive_run_dir is not None else '<not synced>'}")
print(f"Google Drive ZIP 路径 = {drive_zip_path if drive_zip_path is not None else '<not synced>'}")